[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/magrilu/cv-dojo/blob/main/notebooks/camera/camera.ipynb)

# The perspective camera

We will build a geometric model of a photograph as a projection
from three-dimensional projective space to the image plane,

$$
\mathbb P^3 \dashrightarrow \mathbb P^2 .
$$

In practice, a real camera forms an image through a much richer physical process. Light passes
through a system of lenses, reaches a sensor, and is eventually converted into an
array of pixel values. We will leave all of this aside for now and concentrate on the geometry of image formation alone.

That geometry is much older than photography. Its roots reach back to ancient optics, and during the Renaissance the same ideas became the basis of linear perspective: a systematic way of representing a three-dimensional scene on a
two-dimensional surface. Leon Battista Alberti described an image as an
*intersegazione della piramide visiva*, namely a cross-section of the visual pyramid.

> «Sarà adunque pittura non altro che intersegazione della pirramide visiva,
> sicondo data distanza, posto il centro e constituiti i lumi, in una certa
> superficie con linee e colori artificiose representata.»
>
> Painting, then, will be nothing other than the intersection of the visual
> pyramid, at a given distance, with the centre placed and the lights
> established, represented by art with lines and colours on a given surface.
>
> — Alberti, *De pictura*.

::: {.column-margin}
![Sarebbe cosa lunga, difficile e oscura […] seguire ogni cosa con la regola de' matematici.](../../images/leon_battista.jpg){width=100%}
::: 

We will take that construction as our starting point: choose a centre of projection, join it to the points of the scene, and intersect the resulting visual rays with an image plane.

From there we will gradually build the pinhole camera model. Homogeneous coordinates will let us encode the entire projection in a single $3\times4$
matrix,

$$
\mathbf x \sim \mathsf P\,\mathbf X ,
$$

defined up to an arbitrary non-zero scale and therefore carrying **eleven degrees
of freedom**.

In [ ]:
#| echo: false
import sys, subprocess, json
from pathlib import Path

if "google.colab" in sys.modules and not Path("cv-dojo").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/magrilu/cv-dojo.git"], check=True)
    import os
    os.chdir("cv-dojo")

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "cvdojo").exists():
        sys.path.insert(0, str(parent / "src"))
        ROOT = parent
        break

import numpy as np
import matplotlib.pyplot as plt

from cvdojo.house import load_model
from cvdojo.plotting import ACCENT
from cvdojo.scene import set_axes_equal

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": False})

BLUE, GREY, RED = "#3288BD", "0.45", "#C0392B"
DARK = "0.20"   # the world reference frame, which belongs to neither

# One convention, fixed here and used in every figure of the notebook:
#   BLUE   the object,
#   GREY   the rays, and anything shown for reference only,
#   ACCENT the apparatus of projection - the centre, the image plane, the
#          camera frame,
#   DARK   the world reference frame.
EYE_COLOR, EYE_MARKER, EYE_SIZE = ACCENT, "o", 95

model = load_model()
V3 = {k: np.array(v, float) for k, v in model["vertices"].items()}
IDS = list(V3)
EDGES = model["edges"]
index = {k: i for i, k in enumerate(IDS)}

# Geometry convention used throughout the notebook:
# one point = one column, hence X has shape 3 x N.
X = np.column_stack([V3[k] for k in IDS])


In [ ]:
#| echo: false
# ---------------------------------------------------------------------------
# Plotting and scene helpers. The geometric code used in the exposition is
# introduced explicitly in the visible cells below.
# ---------------------------------------------------------------------------


def draw_eye(ax, centre, label=None, offset=None):
    """Draw the centre of projection in either a 2D or a 3D axes."""
    c = np.asarray(centre, float).reshape(-1)
    ax.scatter(*c, marker=EYE_MARKER, s=EYE_SIZE, color=EYE_COLOR,
               edgecolors="white", linewidths=1.2, zorder=8)
    if label:
        off = np.zeros_like(c) if offset is None else np.asarray(offset, float).reshape(-1)
        ax.text(*(c + off), label, color=EYE_COLOR, fontsize=12)


def frame_3d(ax, origin, Rt, labels, color=ACCENT, length=2.0, lw=2.0,
             alpha=1.0, ls="-"):
    """Draw a right-handed frame; Rt stores its axes as columns."""
    o = np.asarray(origin, float).reshape(3)
    Rt = np.asarray(Rt, float)
    for j, name in enumerate(labels):
        end = o + length * Rt[:, j]
        ax.plot(*zip(o, end), lw=lw, color=color, alpha=alpha, ls=ls)
        if name:
            ax.text(*end, name, fontsize=10, color=color, alpha=alpha)


def wireframe_3d(ax, pts, color=BLUE, lw=1.8, dots=True, alpha=1.0):
    """Draw 3D points stored as columns of a 3 x N array."""
    pts = np.asarray(pts, float)
    for a, b in EDGES:
        q = pts[:, [index[a], index[b]]]
        ax.plot(q[0, :], q[1, :], q[2, :], lw=lw, color=color, alpha=alpha)
    if dots:
        ax.scatter(pts[0, :], pts[1, :], pts[2, :], s=18, color=color,
                   alpha=alpha)


def wireframe_2d(ax, xy_, title=None, width=None, height=None, color=BLUE,
                 lw=2.0, image=True, alpha=1.0, dots=True):
    """Draw 2D points stored as columns of a 2 x N array."""
    xy_ = np.asarray(xy_, float)
    for a, b in EDGES:
        q = xy_[:, [index[a], index[b]]]
        ax.plot(q[0, :], q[1, :], lw=lw, color=color, alpha=alpha)
    if dots:
        ax.scatter(xy_[0, :], xy_[1, :], s=20, color=color, alpha=alpha)
    if width is not None:
        ax.set_xlim(0, width)
        ax.set_ylim(height, 0)
    elif image and not ax.yaxis_inverted():
        ax.invert_yaxis()
    ax.set_aspect("equal")
    if title:
        ax.set_title(title, fontsize=10)


def image_frame(ax, width, height, color=GREY):
    """Draw the border of a digital image in pixel coordinates."""
    ax.plot([0, width, width, 0, 0], [0, 0, height, height, 0],
            lw=1.0, ls="--", color=color)


In [ ]:
#| echo: false
# Camera-coordinate view used only by the opening figures.  We construct it
# here without introducing the public extrinsic variables C, R, and t; those
# enter explicitly only when the world reference frame is introduced below.
_C_opening = np.array([[7.0], [-14.0], [7.0]])
_target_opening = X.mean(axis=1, keepdims=True)

_z_opening = _target_opening - _C_opening
_z_opening /= np.linalg.norm(_z_opening)
_x_opening = np.cross(_z_opening[:, 0], np.array([0.0, 0.0, 1.0]))[:, None]
_x_opening /= np.linalg.norm(_x_opening)
_y_opening = np.cross(_z_opening[:, 0], _x_opening[:, 0])[:, None]

_R_opening = np.vstack((_x_opening.T, _y_opening.T, _z_opening.T))
Xc = _R_opening @ (X - _C_opening)

## The visual pyramid

Let $\{\mathbf{X}_i\}$ be the collection of 3D vertices that represents
the Origami House, and consider a **centre of projection** $\mathbf{C}\in \mathbb{P}^3$: Alberti's
eye, or the optical centre of our ideal camera.

Join $\mathbf{C}$ to every point of the scene. The resulting family of concurrent
lines is the **visual pyramid**. To form an image, place a plane $\phi$ across
these rays.

In [ ]:
#| echo: false
#| column: page
#| label: fig-visual-pyramid
#| fig-cap: >-
#|   The visual pyramid. **Left:** rays from the centre of projection through
#|   the vertices of the Origami House intersect an image plane. **Right:** the
#|   resulting intersection points form the image.
plane_z = 0.7 * Xc[2, :].min()
hit = plane_z * Xc[:2, :] / Xc[2:3, :]

fig = plt.figure(figsize=(14.5, 6.0), layout="constrained")
gs = fig.add_gridspec(1, 2, width_ratios=[1.25, 1.0])

ax = fig.add_subplot(gs[0], projection="3d")
for q in Xc.T:
    ax.plot(*zip(np.zeros(3), q), lw=0.7, color=GREY, alpha=.85)
wireframe_3d(ax, Xc, dots=False)
half = 1.45 * np.abs(hit).max()
gx, gy = np.meshgrid([-half, half], [-half, half])
ax.plot_surface(gx, gy, np.full_like(gx, plane_z), alpha=.16, color=ACCENT)
for a, b in EDGES:
    q = hit[:, [index[a], index[b]]]
    ax.plot(q[0, :], q[1, :], np.full(2, plane_z), lw=1.5, color=ACCENT)
draw_eye(ax, np.zeros(3))
ax.set_xlabel("$x_c$"); ax.set_ylabel("$y_c$"); ax.set_zlabel("$z_c$")
ax.set_title("the pyramid, and a plane put in its way", fontsize=10)
ax.view_init(elev=11, azim=-67); set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.25)

ax = fig.add_subplot(gs[1])
wireframe_2d(ax, hit, "the intersection: a picture")
ax.axis("off")
plt.show()


Each ray meets the plane in one image point 
$$
\mathbf x_i = \overline{\mathbf C\mathbf X_i}\cap\phi .
$$

Point by point, the three-dimensional scene is cut by the image plane into a
two-dimensional picture: Alberti's *intersegazione della piramide visiva*.

## From the visual pyramid to coordinates

We can describe this geometric construction algebraically. Let's attach an euclidean reference frame $\{x_{c},y_{c}, z_{c}\}$ to the
camera. Place its origin at the centre of projection $\mathbf C$ and choose the
$z_c$-axis perpendicular to the image plane $\phi$. The line through $\mathbf C$ in
this direction is called the **optical axis**. The $x_c$- and $y_c$-axes run
parallel to the image plane.

If the image plane is at distance $f$ from the centre, the equation of $\phi$ is simply

$$
\phi : z_c=f.
$$

Consider a scene point expressed in the camera reference frame, $\mathbf X_c=(X_c,Y_c,Z_c)^\top$, the **visual ray** through $\mathbf X_c$ from $\mathbf C$ intersects the image plane at
$\mathbf x=(x,y,f)^\top$.

In [ ]:
#| echo: false
#| column: page
#| label: fig-camera-frame
#| fig-cap: >-
#|   Perspective projection in the camera reference frame. **Left:** a scene
#|   point and its visual ray intersect the image plane $z_c=f$. **Right:** in
#|   an $x_cz_c$ section, the projection equation follows from similar
#|   triangles.

# Local names throughout this cell, so that nothing defined earlier is disturbed.
f_d, Zd, Xd, Yd = 1.4, 4.2, 1.55, 1.05
xd, yd = f_d * Xd / Zd, f_d * Yd / Zd

fig = plt.figure(figsize=(11.0, 4.8), layout="constrained")
gs = fig.add_gridspec(1, 2, width_ratios=[1.25, 1.0])

# ------------------------------------------------ the frame, in 3D ----------
ax = fig.add_subplot(gs[0], projection="3d")

hx, hy = 1.15, 0.95
gx, gy = np.meshgrid([-hx, hx], [-hy, hy])
ax.plot_surface(gx, gy, np.full_like(gx, f_d), color=ACCENT, alpha=.14,
                shade=False)
corners = np.array([[-hx, -hy, f_d], [hx, -hy, f_d], [hx, hy, f_d],
                    [-hx, hy, f_d], [-hx, -hy, f_d]])
ax.plot(corners[:, 0], corners[:, 1], corners[:, 2], color=ACCENT, lw=1.2,
        alpha=.8)

ax.plot([0, Xd], [0, Yd], [0, Zd], color=BLUE, lw=1.6)
ax.scatter([Xd], [Yd], [Zd], s=38, color=BLUE, depthshade=False, zorder=5)
ax.scatter([xd], [yd], [f_d], s=34, color=ACCENT, depthshade=False, zorder=5)
draw_eye(ax, np.zeros(3))

frame_3d(ax, np.zeros(3), np.eye(3), [r"$x_c$", r"$y_c$", r"$z_c$"],
         color=GREY, length=1.05, lw=1.1)
ax.plot([0, 0], [0, 0], [0, f_d], color=GREY, lw=.9, ls="--", alpha=.8)
ax.text(-.13, -.10, .52*f_d, r"$f$", fontsize=11, color=GREY)
ax.text(-.10, -.10, -.28, r"$\mathbf{C}$", fontsize=12, color=ACCENT)
ax.text(Xd + .10, Yd + .04, Zd, r"$\mathbf{X}_c$", fontsize=12, color=BLUE)
ax.text(xd + .12, yd + .03, f_d, r"$\mathbf{x}$", fontsize=12, color=ACCENT)
ax.text(-hx, hy, f_d + .06, r"$\phi$", fontsize=14, color=ACCENT)

ax.view_init(elev=20, azim=-61)
ax.set_xlim(-1.15, 1.9); ax.set_ylim(-1.15, 1.55); ax.set_zlim(-.15, 4.7)
ax.set_box_aspect((1.05, 1.0, 1.45))
ax.set_axis_off()
ax.set_title("camera reference frame", fontsize=10)

# ------------------------------------------------ the two sections ----------
def draw_section(ax, coord, image_coord, coord_name, image_name):
    ax.axhline(0, lw=.8, color=GREY, alpha=.7)
    top = 1.20 * max(abs(coord), abs(image_coord))
    ax.plot([f_d, f_d], [-.15, top], lw=1.7, color=ACCENT, alpha=.8)
    ax.plot([0, Zd], [0, coord], lw=1.5, color=BLUE)
    ax.plot([Zd, Zd], [0, coord], lw=.9, ls="--", color=GREY, alpha=.7)
    ax.plot([f_d, f_d], [0, image_coord], lw=2.4, color=ACCENT)
    ax.scatter([Zd, f_d], [coord, image_coord], s=[34, 34],
               color=[BLUE, ACCENT], zorder=5)
    draw_eye(ax, [0, 0])

    ax.text(-.10, -.09, r"$\mathbf{C}$", ha="right", va="top", fontsize=11,
            color=ACCENT)
    ax.text(Zd + .08, coord, r"$\mathbf{X}_c$", ha="left", va="center",
            fontsize=11, color=BLUE)
    ax.text(f_d + .08, image_coord, r"$\mathbf{x}$", ha="left", va="center",
            fontsize=11, color=ACCENT)
    ax.text(f_d, -.09, r"$f$", ha="center", va="top", fontsize=10)
    ax.text(Zd, -.09, r"$Z_c$", ha="center", va="top", fontsize=10)
    ax.text(Zd - .10, .52*coord, rf"${coord_name}$", ha="right", va="center",
            fontsize=10, color=BLUE)
    ax.text(f_d - .09, .52*image_coord, rf"${image_name}$", ha="right",
            va="center", fontsize=10, color=ACCENT)
    ax.text(f_d, top*1.02, r"$\phi$", ha="center", va="bottom", fontsize=12,
            color=ACCENT)

    ax.set_xlim(-.25, Zd + .65); ax.set_ylim(-.25, max(coord, image_coord)*1.25)
    ax.set_xlabel(r"$z_c$")
    ax.set_aspect("equal", adjustable="box")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])


ax = fig.add_subplot(gs[1])
draw_section(ax, Xd, xd, "X_c", "x")
ax.set_ylabel(r"$x_c$")
ax.set_title(r"section in the $x_cz_c$-plane", fontsize=10)
plt.show()

In the $x_cz_c$- and $y_cz_c$-sections, the visual ray gives two pairs of
similar triangles. Hence

$$
\frac{x}{X_c}=\frac{f}{Z_c},
\qquad
\frac{y}{Y_c}=\frac{f}{Z_c},
$$

and therefore

$$
x=f\frac{X_c}{Z_c},
\qquad
y=f\frac{Y_c}{Z_c}.
$$

These are the **perspective projection equations**.

## Depth changes apparent size

The division by $Z_c$ is responsible for one of the most familiar effects of
perspective.
Take two identical Origami Houses, place them side by side at the same depth,
and then move one farther away along the viewing direction. Their 3D size is
unchanged, but in the image the farther house appears smaller. Having a greater
depth $Z_c$, it occupies a smaller region of the image plane.

In [ ]:
#| echo: false
#| column: page
#| label: fig-depth-and-size
#| fig-cap: >-
#|   Two identical Origami Houses at different depths. **Left:** in camera
#|   coordinates, where the two copies have exactly the same size. **Right:**
#|   their projections onto the plane $z_c=f$, where the copy at greater depth
#|   appears smaller.
f_two = 0.7 * Xc[2, :].min()
house_width = np.ptp(Xc[0, :])

near = Xc + np.array([[-0.70 * house_width], [0.0], [0.0]])
far = Xc + np.array([[1.05 * house_width], [0.0], [7.0]])

def to_plane(pts):
    return f_two * pts[:2, :] / pts[2:3, :]

fig = plt.figure(figsize=(13.5, 5.4), layout="constrained")

ax = fig.add_subplot(121, projection="3d")
wireframe_3d(ax, near, dots=False)
wireframe_3d(ax, far, dots=False, alpha=.40)
draw_eye(ax, np.zeros(3), r"$\mathbf{C}$", offset=[0, 0, -2.0])
ax.set_xlabel("$x_c$"); ax.set_ylabel("$y_c$"); ax.set_zlabel("$z_c$")
ax.set_title("same size, different depth", fontsize=10)
ax.view_init(elev=18, azim=-62); set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.2)

ax = fig.add_subplot(122)
wireframe_2d(ax, to_plane(near), None, lw=1.9, dots=False)
wireframe_2d(ax, to_plane(far), None, lw=1.9, alpha=.40, dots=False)
ax.set_title("their images on the plane $z_c=f$", fontsize=10)
ax.axis("off")
plt.show()


This factor $1/Z_c$ also makes
perspective projection nonlinear in Cartesian coordinates.
Homogeneous coordinates let us represent the same geometry with a linear map.

## Homogeneous coordinates make projection linear


So far, a point in the camera frame has been described by its Cartesian
coordinates $(X_c,Y_c,Z_c)$. We now explicitly work in projective spaces, and add one extra coordinate and write

$$
\mathbf X_c=(X_c,Y_c,Z_c,1)^\top,
$$

while an image point is represented by

$$
\mathbf x=(x,y,1)^\top.
$$

Consider the $3\times4$ matrix

$$
\mathsf P_f=
\begin{bmatrix}
f&0&0&0\\
0&f&0&0\\
0&0&1&0
\end{bmatrix}.
$$

Then

$$
\mathsf P_f\mathbf X_c
=
\begin{bmatrix}
fX_c\\
fY_c\\
Z_c
\end{bmatrix}
=
Z_c
\begin{bmatrix}
fX_c/Z_c\\
fY_c/Z_c\\
1
\end{bmatrix}
=
Z_c\,\mathbf x.
$$

Hence perspective projection becomes

$$
\mathbf x \sim \mathsf P_f\mathbf X_c,
$$

where $\sim$ denotes equality up to a non-zero scale.

The division by depth has not disappeared. In homogeneous coordinates the
mapping is linear; the division appears only when we return to Cartesian image
coordinates by normalising the last coordinate to one.

In [ ]:
# Focal length and the corresponding camera matrix.
f = 1.4

P_f = np.array([
    [f,   0.0, 0.0, 0.0],
    [0.0, f,   0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
])

# Add one homogeneous coordinate to every 3D point.
Xh = np.vstack((Xc, np.ones((1, Xc.shape[1]))))

# Linear projection: each column is now a homogeneous image point.
xh = P_f @ Xh

# Dehomogenize each column by dividing by its last coordinate.
xy = xh[:2, :] / xh[2:3, :]

# Look at the resulting image.
fig, ax = plt.subplots(figsize=(5.5, 4.5))
wireframe_2d(ax, xy, None)
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title(r"projection by $\mathsf{P}_f$", fontsize=10)
plt.show()


We can collect the same operation in a small function that will be used throughout the notebook.

In [ ]:
def project_points(X, P):
    """Project 3D points stored as columns through a 3 x 4 camera matrix."""

    # Add one homogeneous coordinate to every 3D point: 3 x N -> 4 x N.
    Xh = np.vstack((X, np.ones((1, X.shape[1]))))

    # Apply the projective camera: each column is a homogeneous image point.
    xh = P @ Xh

    # Dehomogenize each column by dividing by its last coordinate.
    return xh[:2, :] / xh[2:3, :]


> **A historical note.**
> Homogeneous coordinates did not appear all at once in their modern form.
> In 1827  Ferdinand Möbius introduced homogeneous barycentric
> coordinates in his *Der barycentrische Calcul*; Feuerbach arrived at a
> related construction independently at about the same time. Julius Plücker
> subsequently developed and systematized homogeneous coordinate methods for
> projective geometry, notably for lines.
>
> The notation has changed, but the underlying idea is already there: a point
> is represented by a tuple of numbers defined only up to a common non-zero
> scale.

### The camera centre

A projective camera is usually written as a map

$$
\mathsf P:\mathbb P^3\dashrightarrow\mathbb P^2.
$$

The dashed arrow is deliberate: the camera is not defined at every point of
$\mathbb P^3$.

Indeed, a rank-three $3\times4$ camera matrix has a one-dimensional right null
space. Let $\mathbf C$ be its non-zero generator,

$$
\mathsf P\mathbf C=\mathbf 0.
$$

But the zero vector does not represent a point of projective space, so
$\mathbf C$ has no image in $\mathbb P^2$. This exceptional point is precisely
the **camera centre**.

For our simple camera $\mathsf P_f$,

$$
\ker\mathsf P_f
=
\operatorname{span}
\begin{bmatrix}
0\\0\\0\\1
\end{bmatrix},
$$

which is the origin of the camera reference frame, exactly where we placed the
centre of projection.

Thus a projective camera is more precisely the map

$$
\mathbb P^3\setminus\{\mathbf C\}
\longrightarrow
\mathbb P^2,
\qquad
\mathbf X\longmapsto \mathsf P\mathbf X.
$$

Every point other than $\mathbf C$ determines an image point.

Note that the centre is also independent of the arbitrary scale used to represent the camera matrix. If $\alpha\neq0$, then

$$
\ker(\alpha\mathsf P)=\ker(\mathsf P).
$$

Thus $\mathsf P$ and $\alpha\mathsf P$ represent the same projective camera
and have the same camera centre.

In [ ]:
# The camera centre is the right null vector of P_f.
_, _, Vt = np.linalg.svd(P_f)
C_f = Vt[-1, :].reshape(4, 1)

# Choose the representative whose last homogeneous coordinate is one.
C_f = C_f / C_f[-1, 0]

print("camera centre:")
print(C_f)
print("\nP_f @ C:")
print(P_f @ C_f)


## Placing the camera in a world reference frame

So far our reference frame has been attached to the camera. The centre of
projection was the origin, the $z_c$-axis was the optical axis, and the image
plane was simply $z_c=f$.
We now introduce a separate **world reference frame**. This is useful, for
example, when several views of the same scene must be related within a common
coordinate system. For the Origami House, for
example, we can attach a frame to the house, with $z_w$ pointing vertically.

In practice, to project a world point $\mathbf X_w$, we first convert its coordinates to the
camera frame.
Let $\mathbf C$ be the camera centre in world coordinates. The vector

$$
\mathbf X_w-\mathbf C
$$

points from the camera centre to the scene point, but its components are still
measured along the world axes.

Let $\mathbf r_1^\top$, $\mathbf r_2^\top$, and $\mathbf r_3^\top$ be the three axes of the camera frame, expressed in world coordinates. The coordinates of the vector in the camera frame are its projections onto these axes:

$$
\mathbf X_c
=
\begin{bmatrix}
\mathbf r_1^\top(\mathbf X_w-\mathbf C)\\
\mathbf r_2^\top(\mathbf X_w-\mathbf C)\\
\mathbf r_3^\top(\mathbf X_w-\mathbf C)
\end{bmatrix}.
$$

Stacking the three camera axes as the rows of a rotation matrix,

$$
\mathsf R=
\begin{bmatrix}
\mathbf r_1^\top\\
\mathbf r_2^\top\\
\mathbf r_3^\top
\end{bmatrix},
$$

gives

$$
\mathbf X_c=\mathsf R(\mathbf X_w-\mathbf C).
$$

Expanding the expression,

$$
\mathbf X_c
=
\mathsf R\mathbf X_w+\mathbf t,
\qquad
\mathbf t=-\mathsf R\mathbf C.
$$

Thus the world-to-camera change of coordinates is the rigid motion

$$
\begin{bmatrix}
\mathbf X_c\\
1
\end{bmatrix}
=
\begin{bmatrix}
\mathsf R & \mathbf t\\
\mathbf 0^\top & 1
\end{bmatrix}
\begin{bmatrix}
\mathbf X_w\\
1
\end{bmatrix}.
$$

The pair $(\mathsf R,\mathbf t)$ gives the **extrinsic parameters** of the
camera. The rotation describes the orientation of the camera axes, while
$\mathbf t=-\mathsf R\mathbf C$ is the world origin expressed in camera
coordinates.

For our synthetic example we choose the camera centre and orient the camera
towards the Origami House. The following small helper constructs such an
orientation explicitly from the camera axes.

In [ ]:
def look_at(centre, target, up=np.array([[0.0], [0.0], [1.0]])):
    """World-to-camera rotation for a camera looking towards `target`."""

    # Camera z-axis: viewing direction, expressed in world coordinates.
    z_axis = target - centre
    z_axis = z_axis / np.linalg.norm(z_axis)

    # Camera x-axis: orthogonal to the viewing direction and to the reference up.
    x_axis = np.cross(z_axis[:, 0], up[:, 0])[:, None]
    x_axis = x_axis / np.linalg.norm(x_axis)

    # Complete a right-handed orthonormal camera frame.
    y_axis = np.cross(z_axis[:, 0], x_axis[:, 0])[:, None]

    # The camera axes are the rows of the world-to-camera rotation.
    return np.vstack((x_axis.T, y_axis.T, z_axis.T))

We can now choose a concrete camera pose for the synthetic scene. We specify
its centre $\mathbf C$ in world coordinates, orient it towards the Origami
House, and obtain $\mathbf t$ from $\mathbf t=-\mathsf R\mathbf C$.

In [ ]:
# Camera centre in world coordinates.
C = np.array([
    [  7.0],
    [-14.0],
    [  7.0],
])

# Orient the camera towards the centre of the Origami House.
target = X.mean(axis=1, keepdims=True)
R = look_at(C, target)

# Translation of the world origin in camera coordinates.
t = -R @ C

# Express the complete house in the camera frame.
Xc = R @ X + t

print("camera centre C:")
print(np.round(C, 3))
print("\ntranslation t:")
print(np.round(t, 3))

In [ ]:
#| echo: false
#| column: page
#| label: fig-two-frames
#| fig-cap: >-
#|   The world frame is attached to the Origami House, while the camera frame
#|   is centred at $\mathbf C$. The same scene point can be described in either
#|   reference frame.
key = IDS[len(IDS)//2]
Pw = X[:, [index[key]]]
p = Pw[:, 0]

fig = plt.figure(figsize=(14.5, 6.4), layout="constrained")
ax = fig.add_subplot(111, projection="3d")

wireframe_3d(ax, X, dots=False)
frame_3d(ax, np.zeros(3), np.eye(3),
         [r"$x_w$", r"$y_w$", r"$z_w$"], color=DARK, length=12.0)
ax.text(-1.0, 0.6, -1.8, r"$O_w$", fontsize=11, color=DARK)
frame_3d(ax, C, R.T, [r"$x_c$", r"$y_c$", r"$z_c$"], color=ACCENT, length=5.0)
draw_eye(ax, C, r"$\mathbf{C}$", offset=[0, 0, -2.2])

# Highlight one scene point and the vector from the camera centre to it.
ax.scatter(*p, s=70, color=RED, zorder=9)
ax.text(*(p + np.array([0.4, 0.4, 0.6])), r"$\mathbf{X}$", fontsize=12,
        color=RED)
seg = np.hstack((C, Pw))
ax.plot(seg[0, :], seg[1, :], seg[2, :], lw=1.4, ls="--", color=ACCENT)

ax.set_xlabel("$X_w$ [cm]"); ax.set_ylabel("$Y_w$ [cm]")
ax.set_zlabel("$Z_w$ [cm]")
ax.view_init(elev=20, azim=-38); set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.30)
plt.show()


In [ ]:
# Pick one vertex of the house. It is stored as a 3 x 1 column vector.
key = IDS[len(IDS) // 2]
Xw = X[:, [index[key]]]

# Vector from the camera centre to the point, still expressed in world axes.
v = Xw - C

# The rows of R are the camera axes expressed in world coordinates.
x_axis = R[0:1, :]
y_axis = R[1:2, :]
z_axis = R[2:3, :]

# Camera coordinates are the three components of v along those axes.
Xc_direct = R @ v
Xc_from_axes = np.vstack((
    x_axis @ v,
    y_axis @ v,
    z_axis @ v,
))

print("camera coordinates:")
print(np.round(Xc_direct, 3))
print("\nfrom the three axis projections:")
print(np.round(Xc_from_axes, 3))

In [ ]:
def rigid_transform(R, t):
    """Homogeneous matrix of the rigid transformation X' = R X + t."""

    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3:4] = t
    return T


# Assemble the world-to-camera rigid transformation.
T_wc = rigid_transform(R, t)

# Apply it to the same point in homogeneous coordinates.
Xw_h = np.vstack((Xw, [[1.0]]))
Xc_h = T_wc @ Xw_h

print("from R (X_w - C):")
print(np.round(Xc_direct, 3))
print("\nfrom the homogeneous rigid transformation:")
print(np.round(Xc_h[:3], 3))

Projection is now the composition of two maps,

$$
\mathbf X_w \longmapsto \mathbf X_c \longmapsto \mathbf x.
$$

Let

$$
\mathsf T_{wc}=
\begin{bmatrix}
\mathsf R&\mathbf t\\
\mathbf 0^\top&1
\end{bmatrix}.
$$

In homogeneous coordinates,

$$
\lambda\mathbf x
=
\mathsf P_f\,\mathsf T_{wc}
\begin{bmatrix}\mathbf X_w\\1\end{bmatrix},
$$

or, equivalently,

$$
\lambda\mathbf x
=
\begin{bmatrix}
f&0&0\\
0&f&0\\
0&0&1
\end{bmatrix}
[\mathsf R\mid\mathbf t]
\begin{bmatrix}\mathbf X_w\\1\end{bmatrix}.
$$

Thus the same perspective camera that acted on camera coordinates can act
directly on world coordinates through the composed $3\times4$ matrix

$$
\mathsf P=\mathsf P_f\mathsf T_{wc}.
$$

In [ ]:
# Compose the perspective camera with the world-to-camera transformation.
P = P_f @ T_wc

# Project the Origami House directly from world coordinates.
xy = project_points(X, P)

The camera centre can now be recovered using the characterization introduced
earlier. In homogeneous world coordinates, let

$$
\mathbf C_h=
\begin{bmatrix}
\mathbf C\\
1
\end{bmatrix}.
$$

The centre belongs to the right null space of $\mathsf P$. The left $3\times3$
factor in the expression above is invertible, so this is equivalent to

$$
[\mathsf R\mid\mathbf t]\mathbf C_h=\mathbf 0.
$$

Therefore

$$
\mathsf R\mathbf C+\mathbf t=\mathbf 0,
$$

and, since $\mathsf R^{-1}=\mathsf R^\top$,

$$
\mathbf C=-\mathsf R^\top\mathbf t.
$$

The projective null-space characterization and the Euclidean extrinsic parameters therefore identify the same camera centre.

In [ ]:
# Recover the camera centre from the extrinsic parameters.
C_from_extrinsics = -R.T @ t

# In homogeneous coordinates it must lie in the null space of [R | t].
C_h = np.vstack((C_from_extrinsics, [[1.0]]))
Rt = np.hstack((R, t))

print("camera centre recovered from R and t:")
print(np.round(C_from_extrinsics, 6))
print("\n[R | t] C_h:")
print(np.round(Rt @ C_h, 12))

### Rotating the camera

Keep the camera centre $\mathbf C$ fixed and change only its orientation.
If $\mathsf Q$ is a rotation expressed in the camera frame, the new
world-to-camera rotation is

$$
\mathsf R'=\mathsf Q\mathsf R.
$$

Since the centre has not moved, the corresponding translation is still
determined by $\mathbf t'=-\mathsf R'\mathbf C.$

In [ ]:
# Define Q as a rotation of angle theta around the y axis
theta = np.deg2rad(15.0)

Q = np.array([
    [ np.cos(theta), 0.0, np.sin(theta)],
    [           0.0, 1.0,           0.0],
    [-np.sin(theta), 0.0, np.cos(theta)],
])

R_rot = Q @ R
t_rot = -R_rot @ C

P_rot = P_f @ rigid_transform(R_rot, t_rot)
xy_rot = project_points(X, P_rot)


In [ ]:
#| echo: false
#| column: page
#| label: fig-camera-rotation
#| fig-cap: >-
#|   Rotating the camera about its centre. The camera centre remains fixed,
#|   while its orientation changes from $\mathsf R$ to $\mathsf R'$. The two
#|   right panels show the corresponding perspective projections of the same
#|   fixed Origami House.
fig = plt.figure(figsize=(12.5, 4.6), layout="constrained")
gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 1.0, 1.0])

# Camera pose in the world
ax = fig.add_subplot(gs[0], projection="3d")

# The Origami House stays fixed in the world.
wireframe_3d(ax, X, dots=False)

# Original camera frame, shown as reference.
frame_3d(
    ax, C, R.T, ["", "", ""],
    color=GREY, length=4.5, ls="--"
)

# Camera frame after the rotation.
frame_3d(
    ax, C, R_rot.T,
    [r"$x_c'$", r"$y_c'$", r"$z_c'$"],
    color=ACCENT, length=4.5
)

draw_eye(ax, C, r"$\mathbf{C}$", offset=[0, 0, -2.0])

ax.view_init(elev=18, azim=-52)
set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.35)
ax.set_axis_off()

# Use the same coordinate limits before and after the rotation.
all_xy = np.hstack((xy, xy_rot))
lo = all_xy.min(axis=1)
hi = all_xy.max(axis=1)
pad = 0.10 * (hi - lo)

# Original projection
ax = fig.add_subplot(gs[1])

# Same 3D wireframe, projected by the original camera.
wireframe_2d(ax, xy, color=GREY)
ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0])
ax.set_ylim(hi[1] + pad[1], lo[1] - pad[1])
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("before")

# Projection after rotating the camera
ax = fig.add_subplot(gs[2])

# Same 3D wireframe, now projected by the rotated camera.
wireframe_2d(ax, xy_rot)
ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0])
ax.set_ylim(hi[1] + pad[1], lo[1] - pad[1])
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("after rotation")

plt.show()

### Translating the camera

Now keep the camera orientation fixed and move its centre from $\mathbf C$ to
$\mathbf C'$. Since the camera axes do not change, $\mathsf R'=\mathsf R.$
The new centre $\mathbf C'$ determines the corresponding translation,
$$
\mathbf t'=-\mathsf R\mathbf C'.
$$

In [ ]:
# Move the camera centre in the world frame.
C_tr = C + np.array([
    [-3.0],
    [ 0.0],
    [ 1.0],
])

# Keep the camera orientation fixed.
R_tr = R
t_tr = -R_tr @ C_tr

# Assemble the translated camera and project the same Origami House.
P_tr = P_f @ rigid_transform(R_tr, t_tr)
xy_tr = project_points(X, P_tr)


In [ ]:
#| echo: false
#| column: page
#| label: fig-camera-translation
#| fig-cap: >-
#|   Translating the camera while keeping its orientation fixed. The camera
#|   axes remain parallel, while the centre moves from $\mathbf C$ to
#|   $\mathbf C'$. The two right panels show the corresponding perspective
#|   projections of the same fixed Origami House.
fig = plt.figure(figsize=(12.5, 4.6), layout="constrained")
gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 1.0, 1.0])

# Camera pose in the world
ax = fig.add_subplot(gs[0], projection="3d")

# The Origami House stays fixed in the world.
wireframe_3d(ax, X, dots=False)

# frame_3d expects frame axes as columns, hence R.T.
# Original camera frame.
frame_3d(
    ax, C, R.T, ["", "", ""],
    color=GREY, length=4.5, ls="--"
)

# Translated camera frame: same axes, different centre.
frame_3d(
    ax, C_tr, R_tr.T,
    [r"$x_c'$", r"$y_c'$", r"$z_c'$"],
    color=ACCENT, length=4.5
)

# Original and translated camera centres.
ax.scatter(*C[:, 0], marker=EYE_MARKER, s=EYE_SIZE,
           color=GREY, zorder=7)
draw_eye(ax, C_tr, r"$\mathbf{C}'$", offset=[0, 0, -2.0])

# Displacement of the camera centre.
ax.plot(
    [C[0, 0], C_tr[0, 0]],
    [C[1, 0], C_tr[1, 0]],
    [C[2, 0], C_tr[2, 0]],
    color=GREY, ls="--", lw=1.2
)

ax.view_init(elev=18, azim=-52)
set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.35)
ax.set_axis_off()

# Use the same coordinate limits for the two projections.
all_xy = np.hstack((xy, xy_tr))
lo = all_xy.min(axis=1)
hi = all_xy.max(axis=1)
pad = 0.10 * (hi - lo)

# Original projection
ax = fig.add_subplot(gs[1])

# Same 3D wireframe, projected by the original camera.
wireframe_2d(ax, xy, color=GREY)
ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0])
ax.set_ylim(hi[1] + pad[1], lo[1] - pad[1])
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("before")

# Projection after translating the camera
ax = fig.add_subplot(gs[2])

# Same 3D wireframe, now projected from the translated camera centre.
wireframe_2d(ax, xy_tr)
ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0])
ax.set_ylim(hi[1] + pad[1], lo[1] - pad[1])
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("after translation")

plt.show()

Rotation changes the orientation of the camera frame. Translation changes its
origin. Together they determine the camera pose with respect to the world.

## From the image plane to pixels

So far we have described projection using continuous coordinates on the image
plane. A digital image instead uses **pixel coordinates**: a column coordinate
$u$ and a row coordinate $v$, usually measured from the upper-left corner.

It is useful to separate the geometry of perspective projection from this choice
of image coordinates. Dividing the camera coordinates by depth gives the
**normalised image coordinates**

$$
\mathbf x_n
=
\begin{bmatrix}
X_c/Z_c\\
Y_c/Z_c\\
1
\end{bmatrix}.
$$

Geometrically, these are the intersections of the visual rays with the reference
plane $z_c=1$.

To convert these coordinates to pixels, we must account for the physical scale
of the sensor and for the choice of pixel coordinate system. If a pixel has
width $p_x$ and height $p_y$, the focal length expressed in pixels is

$$
f_x=\frac{f}{p_x},
\qquad
f_y=\frac{f}{p_y}.
$$

Projection can therefore be viewed as two successive maps,

$$
\mathbf X_c
\longmapsto
\mathbf x_n
\longmapsto
\mathbf x,
$$

from 3D camera coordinates to normalised image coordinates, and then to pixel
coordinates.

### The intrinsic parameters
The change from normalised image coordinates to pixel coordinates is an
**affine transformation**, represented in homogeneous coordinates by

$$
\mathbf x \sim \mathsf K\,\mathbf x_n,
$$

where

$$
\mathsf K=
\begin{bmatrix}
f_x & s   & c_x\\
0   & f_y & c_y\\
0   & 0   & 1
\end{bmatrix}.
$$

The quantities $f_x$ and $f_y$ are the focal length expressed in pixels along
the two image axes. The point $(c_x,c_y)$ is the **principal point**, where the
optical axis intersects the image plane, expressed in pixel coordinates. The parameter $s$ measures the **skew** of the image coordinates: it is zero
when the two pixel axes are orthogonal, and non-zero when they are not. In most
modern digital cameras it is negligible and is usually set to zero.

These quantities are the **intrinsic parameters** of the camera. They describe
the transformation from the normalised image plane to the coordinate system of
the digital image.

The effect of the focal length and the principal point can be seen directly in
the image.

In [ ]:
def intrinsic_matrix(fx, fy=None, cx=0.0, cy=0.0, skew=0.0):
    """Build the 3 x 3 intrinsic calibration matrix."""
    if fy is None:
        fy = fx

    return np.array([
        [fx,  skew, cx],
        [0.0, fy,   cy],
        [0.0, 0.0, 1.0],
    ])


# A 960 x 720 digital image.
W_PIX, H_PIX = 960, 720

# Square pixels, zero skew, principal point at the image centre.
K = intrinsic_matrix(
    fx=850,
    cx=W_PIX / 2,
    cy=H_PIX / 2,
)

# Compose intrinsics and extrinsics into a single 3 x 4 camera matrix.
P = K @ np.hstack((R, t))

# Project the Origami House directly into pixel coordinates.
uv = project_points(X, P)

### Changing the focal length

Keep the camera pose, the image size, and the principal point fixed, and vary
only the focal length. From

$$
u-c_x=f_x\frac{X_c}{Z_c},
\qquad
v-c_y=f_y\frac{Y_c}{Z_c},
$$

increasing $f_x$ and $f_y$ moves image points farther from the principal point.
The projected object therefore appears larger. For a fixed image size, the
field of view becomes narrower.

In [ ]:
# Compare three cameras that differ only in focal length.
focals = [500.0, 850.0, 1400.0]

focal_projections = []

for f_pix in focals:
    K_f = intrinsic_matrix(
        fx=f_pix,
        fy=f_pix,
        cx=W_PIX / 2,
        cy=H_PIX / 2,
    )

    P_focal = K_f @ np.hstack((R, t))
    focal_projections.append(project_points(X, P_focal))

In [ ]:
#| echo: false
#| column: screen-inset
#| label: fig-focal-length
#| fig-cap: >-
#|   Changing the focal length while keeping the camera pose and image size
#|   fixed. Left: the same visual rays intersect image planes at three
#|   different distances from the camera centre. Right: increasing the focal
#|   length magnifies the projection and reduces the field of view.

fig = plt.figure(figsize=(15.5, 4.9), layout="constrained")
gs = fig.add_gridspec(1, 4, width_ratios=[1.5, 1.0, 1.0, 1.0])

# --- The same visual pyramid, sliced at different focal lengths ---------------
ax = fig.add_subplot(gs[0], projection="3d")

for q in Xc.T:
    ax.plot(*zip(np.zeros(3), q), lw=0.6, color=GREY, alpha=0.8)

wireframe_3d(ax, Xc, dots=False)
draw_eye(ax, np.zeros((3, 1)))

# A common scale converts focal lengths in pixels into distances in this
# schematic 3D drawing. Only their relative values matter here.
pixel_pitch = Xc[2, :].min() / max(focals)

for f_pix in focals:
    f_plane = f_pix * pixel_pitch

    # Intersection of each visual ray with the plane z_c = f_plane.
    hit = f_plane * Xc[:2, :] / Xc[2:3, :]

    h = 1.35 * np.abs(hit).max()
    gx, gy = np.meshgrid([-h, h], [-h, h])

    ax.plot_surface(
        gx, gy, np.full_like(gx, f_plane),
        alpha=0.13, color=ACCENT
    )

    # Draw the projected Origami House on this image plane.
    for a, b in EDGES:
        q = hit[:, [index[a], index[b]]]
        ax.plot(
            q[0, :], q[1, :], np.full(2, f_plane),
            lw=1.2, color=ACCENT
        )

ax.set_title("one visual pyramid, three image planes", fontsize=10)
ax.view_init(elev=9, azim=-70)
set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.65)
ax.set_axis_off()

# --- Corresponding digital images --------------------------------------------
for j, (f_pix, uv_focal) in enumerate(zip(focals, focal_projections)):
    ax = fig.add_subplot(gs[j + 1])

    wireframe_2d(
        ax, uv_focal,
        rf"$f_x=f_y={f_pix:.0f}$ px",
        W_PIX, H_PIX,
        dots=False
    )
    image_frame(ax, W_PIX, H_PIX)
    ax.axis("off")

plt.show()

### Changing the principal point

Now keep the focal length and the camera pose fixed, and change only the
principal point. From

$$
u=f_x x_n+c_x,
\qquad
v=f_y y_n+c_y,
$$

a change in $(c_x,c_y)$ adds the same offset to every projected point. The
shape and scale of the projection are unchanged; only its position in pixel
coordinates changes.

In [ ]:
# Compare three cameras that differ only in principal point.
principal_points = [
    (W_PIX / 2,       H_PIX / 2),
    (W_PIX / 2 - 190, H_PIX / 2 - 90),
    (W_PIX / 2 + 150, H_PIX / 2 + 110),
]

principal_point_projections = []

for cx, cy in principal_points:
    K_c = intrinsic_matrix(
        fx=850,
        fy=850,
        cx=cx,
        cy=cy,
    )

    P_c = K_c @ np.hstack((R, t))
    principal_point_projections.append(project_points(X, P_c))

In [ ]:
#| echo: false
#| column: page
#| label: fig-principal-point
#| fig-cap: >-
#|   Changing the principal point while keeping the focal length and camera
#|   pose fixed. The cross marks $(c_x,c_y)$. Changing the principal point
#|   translates the projection in pixel coordinates without changing its
#|   shape or scale.

fig, axes = plt.subplots(
    1, 3,
    figsize=(14.5, 4.4),
    layout="constrained"
)

for ax, (cx, cy), uv_c in zip(
    axes,
    principal_points,
    principal_point_projections,
):
    # Same Origami House, with only the principal point changed.
    wireframe_2d(
        ax,
        uv_c,
        rf"$(c_x,c_y)=({cx:.0f},{cy:.0f})$",
        W_PIX,
        H_PIX,
        dots=False,
    )

    # The principal point is the origin of the normalized image plane
    # expressed in pixel coordinates.
    ax.scatter(
        [cx], [cy],
        marker="+",
        s=140,
        lw=2.0,
        color=ACCENT,
        zorder=6,
    )

    image_frame(ax, W_PIX, H_PIX)
    ax.axis("off")

plt.show()

The principal point also has a nice interpretation in normalised image
coordinates. The optical axis meets the normalised image plane at

$$
\mathbf x_n=
\begin{bmatrix}
0\\
0\\
1
\end{bmatrix}.
$$

Applying the intrinsic transformation gives

$$
\mathsf K
\begin{bmatrix}
0\\
0\\
1
\end{bmatrix}
=
\begin{bmatrix}
c_x\\
c_y\\
1
\end{bmatrix}.
$$

Thus $(c_x,c_y)$ is precisely the pixel coordinate of the point where the
optical axis meets the image plane.

Cropping an image changes the pixel coordinate system and therefore changes
the numerical coordinates of the principal point, even though the underlying
camera geometry has not changed.

## The camera matrix

We can now combine the three steps of perspective projection.

In camera coordinates, normalised projection is

$$
\mathbf x_n
\sim
\begin{bmatrix}
\mathsf I & \mathbf 0
\end{bmatrix}
\begin{bmatrix}
\mathbf X_c\\
1
\end{bmatrix}.
$$

The intrinsic matrix converts normalised image coordinates to pixel
coordinates,

$$
\mathbf x \sim \mathsf K\mathbf x_n.
$$

Finally, if the scene is described in world coordinates,

$$
\begin{bmatrix}
\mathbf X_c\\
1
\end{bmatrix}
=
\begin{bmatrix}
\mathsf R & \mathbf t\\
\mathbf 0^\top & 1
\end{bmatrix}
\begin{bmatrix}
\mathbf X_w\\
1
\end{bmatrix}.
$$

Combining the three maps gives

$$
\mathbf x
\sim
\mathsf K
[\mathsf R\mid\mathbf t]
\begin{bmatrix}
\mathbf X_w\\
1
\end{bmatrix}
$$

or, writing the world point directly in homogeneous coordinates,

$$
\mathbf x\sim\mathsf P\mathbf X_w,
\qquad
\mathsf P=\mathsf K[\mathsf R\mid\mathbf t].
$$

The two factors have different roles. The intrinsic matrix $\mathsf K$
describes the mapping from the normalised image plane to pixel coordinates,
while $(\mathsf R,\mathbf t)$ describes the pose of the camera with respect to
the world frame.

A general intrinsic matrix has five parameters, while a rigid camera pose has
six. The resulting eleven parameters agree with the eleven degrees of freedom
of a $3\times4$ camera matrix defined up to a non-zero scale.

In [ ]:
#| echo: false
#| column: page
#| label: fig-three-maps
#| fig-cap: >-
#|   The stages of perspective projection. Top left: the Origami House and
#|   camera in world coordinates. Top right: the same geometry expressed in
#|   the camera frame. Bottom left: perspective division gives normalised
#|   image coordinates. Bottom right: the intrinsic matrix $\mathsf K$
#|   converts them to pixel coordinates.

# Normalised image coordinates.
p_norm = Xc[:2, :] / Xc[2:3, :]

fig = plt.figure(figsize=(13, 9.6), layout="constrained")

# --- World coordinates -------------------------------------------------------
ax = fig.add_subplot(221, projection="3d")

wireframe_3d(ax, X, dots=False)

# frame_3d expects frame axes as columns, hence R.T.
frame_3d(
    ax, C, R.T,
    [r"$x_c$", r"$y_c$", r"$z_c$"],
    color=ACCENT,
    length=3.0,
)

frame_3d(
    ax, np.zeros((3, 1)), np.eye(3),
    [r"$x_w$", r"$y_w$", r"$z_w$"],
    color=DARK,
    length=3.0,
)

draw_eye(ax, C)

# A few visual rays from the camera centre to the house.
for key in [IDS[0], IDS[-1], IDS[len(IDS) // 2]]:
    segment = np.hstack((C, X[:, [index[key]]]))
    ax.plot(
        segment[0, :],
        segment[1, :],
        segment[2, :],
        lw=0.9,
        color=GREY,
    )

ax.set_title("world coordinates", fontsize=10)
ax.view_init(elev=22, azim=-55)
set_axes_equal(ax)
ax.set_axis_off()


# --- Camera coordinates ------------------------------------------------------
ax = fig.add_subplot(222, projection="3d")

wireframe_3d(ax, Xc, dots=False)

frame_3d(
    ax, np.zeros((3, 1)), np.eye(3),
    [r"$x_c$", r"$y_c$", r"$z_c$"],
    color=ACCENT,
    length=4.0,
)

draw_eye(ax, np.zeros((3, 1)))

ax.set_title(
    r"after $[\mathsf{R}\mid\mathbf{t}]$: camera coordinates",
    fontsize=10,
)

ax.view_init(elev=18, azim=-60)
set_axes_equal(ax)
ax.set_axis_off()


# --- Normalised image coordinates -------------------------------------------
ax = fig.add_subplot(223)

wireframe_2d(
    ax,
    p_norm,
    "normalised image coordinates",
    dots=False,
)

ax.set_xlabel(r"$x_n=X_c/Z_c$")
ax.set_ylabel(r"$y_n=Y_c/Z_c$")


# --- Pixel coordinates -------------------------------------------------------
ax = fig.add_subplot(224)

wireframe_2d(
    ax,
    uv,
    r"after $\mathsf{K}$: pixel coordinates",
    W_PIX,
    H_PIX,
    dots=False,
)

image_frame(ax, W_PIX, H_PIX)

ax.set_xlabel("$u$ [px]")
ax.set_ylabel("$v$ [px]")

plt.show()

## Back-projecting a pixel

The chain of maps can also be followed in the opposite direction.

Forward projection maps a 3D point to a single image point,

$$
\mathbf X_w
\longmapsto
\mathbf X_c
\longmapsto
\mathbf x_n
\longmapsto
\mathbf x.
$$

This map loses depth. Consequently, going backwards from an image point does
not recover a unique point of space: it recovers the line of 3D points that
project to it.

Starting from a pixel

$$
\mathbf x=
\begin{bmatrix}
u\\v\\1
\end{bmatrix},
$$

the inverse intrinsic transformation gives its normalised image coordinates,

$$
\mathbf x_n
\sim
\mathsf K^{-1}\mathbf x.
$$

For zero skew,

$$
\mathsf K^{-1}\mathbf x
=
\begin{bmatrix}
(u-c_x)/f_x\\
(v-c_y)/f_y\\
1
\end{bmatrix}.
$$

In the camera frame this vector gives the direction of the corresponding
visual ray. Rotating the direction back to the world frame gives

$$
\mathbf d_w
\sim
\mathsf R^\top\mathsf K^{-1}\mathbf x.
$$

Since the ray starts at the camera centre $\mathbf C$, its points are

$$
\mathbf X_w(\mu)
=
\mathbf C
+
\mu\,\mathsf R^\top\mathsf K^{-1}\mathbf x,
\qquad
\mu>0.
$$

Thus a pixel determines a direction, but not a depth. A single view does not
determine the value of $\mu$.

In [ ]:
def backproject_rays(pixels, K, R, t):
    """Back-project 2D pixels, stored as columns, to unit rays in the world."""

    # Homogeneous pixel coordinates, one pixel per column.
    x = np.vstack((pixels, np.ones((1, pixels.shape[1]))))

    # Remove the intrinsic transformation:
    # each column is now a ray direction in the camera frame.
    d_camera = np.linalg.solve(K, x)

    # Express the same directions in the world frame.
    d_world = R.T @ d_camera

    # Use unit directions, so the ray parameter measures Euclidean distance.
    d_world /= np.linalg.norm(d_world, axis=0, keepdims=True)

    # Camera centre in world coordinates.
    C = -R.T @ t

    return C, d_world

In [ ]:
# Pick three projected vertices of the Origami House.
picked = [IDS[0], IDS[len(IDS) // 2], IDS[-1]]
picked_idx = [index[k] for k in picked]

pixels = uv[:, picked_idx]

# Back-project them into the world.
origin, directions = backproject_rays(pixels, K, R, t)

In [ ]:
#| echo: false
#| column: page
#| label: fig-backprojection
#| fig-cap: >-
#|   Back-projection of three image points. Left: three pixels in the digital
#|   image. Right: each pixel defines a visual ray through the camera centre.
#|   The original 3D vertices lie on these rays, but their position along the
#|   ray is not determined by a single image.

cols = [BLUE, RED, "#7B3294"]

fig = plt.figure(figsize=(14.0, 5.2), layout="constrained")
gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.15])

# --- Three pixels in the image -----------------------------------------------
ax = fig.add_subplot(gs[0])

wireframe_2d(
    ax,
    uv,
    "three pixels in the image",
    W_PIX,
    H_PIX,
    color=GREY,
    lw=1.4,
    dots=False,
)

image_frame(ax, W_PIX, H_PIX)

for col, key, pixel in zip(cols, picked, pixels.T):
    u, v = pixel

    ax.scatter(
        [u], [v],
        s=90,
        color=col,
        zorder=6,
    )

    ax.text(
        u + 18,
        v - 16,
        key,
        fontsize=10,
        color=col,
    )

ax.axis("off")


# --- Corresponding rays in the world -----------------------------------------
ax = fig.add_subplot(gs[1], projection="3d")

wireframe_3d(ax, X, dots=False)

for j, (col, key) in enumerate(zip(cols, picked)):
    direction = directions[:, [j]]
    vertex = X[:, [index[key]]]

    # Extend the ray slightly beyond the vertex that generated the pixel.
    reach = 1.18 * np.linalg.norm(vertex - origin)
    segment = np.hstack((
        origin,
        origin + reach * direction,
    ))

    ax.plot(
        segment[0, :],
        segment[1, :],
        segment[2, :],
        lw=1.6,
        color=col,
    )

    ax.scatter(
        *vertex[:, 0],
        s=60,
        color=col,
        zorder=7,
    )

draw_eye(
    ax,
    origin,
    r"$\mathbf{C}$",
    offset=[0, 0, -2.0],
)

ax.set_title("their visual rays in the world", fontsize=10)
ax.view_init(elev=18, azim=-52)
set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.65)
ax.set_axis_off()

plt.show()

In [ ]:
# Check that each original vertex lies on its back-projected ray.
for j, key in enumerate(picked):
    d = directions[:, [j]]
    v = X[:, [index[key]]] - origin

    distance_to_ray = np.linalg.norm(v - d * (d.T @ v).item())

    print(f"{key}: distance from ray = {distance_to_ray:.2e} cm")

A calibrated image also determines angles between visual rays. For two pixels,

$$
\mathbf d_i=\mathsf K^{-1}\mathbf x_i,
$$

and therefore

$$
\cos\theta
=
\frac{
\mathbf d_1^\top\mathbf d_2
}{
\lVert\mathbf d_1\rVert
\lVert\mathbf d_2\rVert
}.
$$

The camera pose does not appear: applying the same rotation to both rays does
not change the angle between them.

In [ ]:
# Ray directions obtained from two pixels using calibration alone.
xh = np.vstack((pixels, np.ones((1, pixels.shape[1]))))
d = np.linalg.solve(K, xh)
d /= np.linalg.norm(d, axis=0, keepdims=True)

cos_theta = (d[:, [0]].T @ d[:, [-1]]).item()
theta = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))

print(f"angle between the two rays: {theta:.2f} deg")

### Back-projection from the camera matrix

If only the camera matrix $\mathsf P$ is known, back-projection is still
possible, but the result is naturally projective.

The camera centre is the right null space of $\mathsf P$,

$$
\mathsf P\mathbf C=\mathbf 0.
$$

For an image point $\mathbf x$, the pseudoinverse provides one 3D point

$$
\mathbf X_0=\mathsf P^+\mathbf x.
$$

Since a camera matrix has rank three,

$$
\mathsf P\mathsf P^+=\mathsf I,
$$

and therefore

$$
\mathsf P\mathbf X_0=\mathbf x.
$$

Adding any multiple of the camera centre does not change the image,

$$
\mathsf P(\mathbf X_0+\mu\mathbf C)
=
\mathbf x.
$$

Hence the back-projection of $\mathbf x$ is the projective line

$$
\mathbf X
\sim
\alpha\,\mathsf P^+\mathbf x
+
\beta\,\mathbf C,
\qquad \alpha\neq0.
$$

Its projective closure contains the camera centre $\mathbf C$, where the camera
map itself is not defined.

The pseudoinverse is therefore not an inverse camera: it selects one point on
the back-projected line, while the null space determines the remaining
ambiguity.

## Questions to leave open

**Can changing the focal length compensate for moving the camera?** Consider a subject at depth
$D$, whose image scale is proportional to $f/D$. Move the camera along its optical axis so that
the subject is now at depth $D'$. How must $f$ change to keep the subject at exactly the same
size in the image? Now consider a second object at a different depth $Z$. Will the same change
of focal length keep its image unchanged as well? This is the geometry behind the **dolly zoom**.

**What happens to points at infinity?** A world direction $\mathbf d$ can be represented by
$\mathbf X_\infty=(\mathbf d^\top,0)^\top$. Project it with
$\mathsf P=\mathsf K[\mathsf R\mid\mathbf t]$. Which camera parameters determine its image, and
which disappear? What does this imply about the vanishing point of a family of parallel lines
when the camera is translated without being rotated?

## Further reading

- Fusiello, A. Computer Vision: *Three-dimensional Reconstruction Techniques*, Springer Cham, 2024. The perspective camera, its intrinsic and extrinsic parameters, and back-projection.
- Hartley, R. and Zisserman, A. *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press, 2004. Chapter 6 for the projective camera and its decomposition.
- Alberti, L. B. *De pictura* / *Della pittura*, 1435–36, for the visual pyramid and the *velo*. Cecil Grayson's is the standard critical edition.
- Andersen, K. *The Geometry of an Art*, Springer, 2007, for the historical development of mathematical perspective.

---

**Luca Magri** — Computer Vision Dojo  
Code MIT · text and figures CC BY-NC-ND 4.0  
<https://magrilu.github.io/cv-dojo/>